In [1]:
import zipfile
import shutil
from pathlib import Path
import csv
import json
import re

import xml.etree.ElementTree as ET

import numpy as np
from PIL import Image
import pytesseract
import cv2


# =========================
# CONFIG
# =========================

# Path to Tesseract executable on Windows.
# Change this if Tesseract is installed somewhere else.
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# OCR / ADA thresholds
OCR_CONF_THRESHOLD = 60        # minimum OCR confidence for counting text
MIN_CONTRAST_PASS = 4.5        # WCAG AA for normal text
MIN_TEXT_HEIGHT_PX = 14        # smallest acceptable text height in pixels (rough)
BLUR_THRESHOLD = 80.0          # variance of Laplacian; lower = blurrier
ASSUMED_PRINT_WIDTH_IN = 6.5   # assume images will be ~6.5 inches wide on the page
MIN_PRINT_DPI = 300            # standard print quality threshold

# Alt text scoring thresholds
ALT_MIN_WORDS = 3
ALT_MAX_CHARS = 200

# =========================
# UTILITIES
# =========================

def srgb_to_linear(c):
    """Convert 0–1 sRGB channel to linear."""
    if c <= 0.04045:
        return c / 12.92
    return ((c + 0.055) / 1.055) ** 2.4

def relative_luminance(rgb):
    """rgb is (R,G,B) in 0–255."""
    r, g, b = [x / 255.0 for x in rgb]
    r_lin = srgb_to_linear(r)
    g_lin = srgb_to_linear(g)
    b_lin = srgb_to_linear(b)
    return 0.2126 * r_lin + 0.7152 * g_lin + 0.0722 * b_lin

def contrast_ratio(c1, c2):
    """Return WCAG contrast ratio between two RGB colors."""
    L1 = relative_luminance(c1)
    L2 = relative_luminance(c2)
    L_light = max(L1, L2)
    L_dark = min(L1, L2)
    return (L_light + 0.05) / (L_dark + 0.05)

def make_json_safe(value):
    """Recursively convert NumPy types to native Python types for JSON."""
    import numpy as np
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, list):
        return [make_json_safe(v) for v in value]
    if isinstance(value, dict):
        return {k: make_json_safe(v) for k, v in value.items()}
    return value


# =========================
# MODULE 1: Extract images & alt text from Word
# =========================

def extract_images(docx_path, out_dir):
    """
    Extract embedded images from word/media/* in a DOCX.
    Returns (list_of_filenames, media_dir_path).
    """
    docx_path = Path(docx_path)
    out_dir = Path(out_dir)
    media_dir = out_dir / "media"
    media_dir.mkdir(parents=True, exist_ok=True)

    extracted = []

    with zipfile.ZipFile(docx_path, 'r') as z:
        for file in z.namelist():
            if file.startswith("word/media/"):
                filename = Path(file).name
                target = media_dir / filename

                with z.open(file) as src, open(target, "wb") as dst:
                    shutil.copyfileobj(src, dst)

                print(f"Extracted: {file} -> {target}")
                extracted.append(filename)

    return extracted, media_dir


def extract_alt_text_map(docx_path):
    """
    Return a dict: { image_filename: alt_text }
    using document.xml and its relationships.
    """
    docx_path = Path(docx_path)
    alt_map = {}

    with zipfile.ZipFile(docx_path, 'r') as z:
        ns = {
            "w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main",
            "wp": "http://schemas.openxmlformats.org/drawingml/2006/wordprocessingDrawing",
            "a":  "http://schemas.openxmlformats.org/drawingml/2006/main",
            "pic": "http://schemas.openxmlformats.org/drawingml/2006/picture"
        }

        # read relationships to map rId -> image file name
        rels = {}
        rels_xml = z.read("word/_rels/document.xml.rels")
        rels_root = ET.fromstring(rels_xml)
        rel_ns = {"": "http://schemas.openxmlformats.org/package/2006/relationships"}

        for rel in rels_root.findall("Relationship", rel_ns):
            if rel.attrib.get("Type", "").endswith("/image"):
                rels[rel.attrib["Id"]] = Path(rel.attrib["Target"]).name

        # parse main document xml for pic elements
        doc_xml = z.read("word/document.xml")
        root = ET.fromstring(doc_xml)

        for pic in root.findall(".//pic:pic", ns):
            cnvpr = pic.find("pic:nvPicPr/pic:cNvPr", ns)
            if cnvpr is None:
                continue

            alt_text = cnvpr.attrib.get("descr", "")
            blip = pic.find(".//a:blip", ns)
            if blip is not None:
                embed = blip.attrib.get("{http://schemas.openxmlformats.org/officeDocument/2006/relationships}embed")
                image_file = rels.get(embed)
                if image_file:
                    alt_map[image_file] = alt_text

    return alt_map


# =========================
# MODULE 2: ADA image analysis (OCR, contrast, text size, blur, DPI, etc.)
# =========================

def simulate_colorblind_deuteranopia(img_np):
    """
    Very simple deuteranopia simulation. Not medically perfect,
    but enough to flag risky color-only encodings.
    """
    # coefficients adapted from common simulation approximations
    r, g, b = img_np[:,:,0].astype(float), img_np[:,:,1].astype(float), img_np[:,:,2].astype(float)
    r2 = 0.625 * r + 0.7 * g + 0.0 * b
    g2 = 0.7   * r + 0.3 * g + 0.0 * b
    b2 = 0.0   * r + 0.3 * g + 1.0 * b

    sim = np.stack([
        np.clip(r2, 0, 255),
        np.clip(g2, 0, 255),
        np.clip(b2, 0, 255)
    ], axis=-1).astype(np.uint8)
    return sim


def estimate_table_like(ocr_boxes):
    """
    Heuristic: if there are many text boxes arranged in rows/columns,
    likely a table rendered as an image.
    """
    if len(ocr_boxes) < 20:
        return False

    ys = [b["bbox"][1] for b in ocr_boxes]
    xs = [b["bbox"][0] for b in ocr_boxes]

    # cluster y positions into rows (tolerant)
    ys_sorted = sorted(ys)
    row_breaks = 1
    for i in range(1, len(ys_sorted)):
        if abs(ys_sorted[i] - ys_sorted[i-1]) > 10:
            row_breaks += 1

    rows = row_breaks
    cols = len(set([round(x/20)*20 for x in xs]))  # crude binning

    # heuristic: table-like if several rows and columns
    return (rows >= 3 and cols >= 3)


def estimate_complex_figure(ocr_boxes):
    """
    Heuristic: complex if a lot of text / labels / numbers.
    """
    total_chars = sum(len(b["text"]) for b in ocr_boxes)
    num_boxes = len(ocr_boxes)
    numeric_tokens = sum(1 for b in ocr_boxes if re.search(r"\d", b["text"]))
    return (total_chars > 100 and num_boxes > 15) or (numeric_tokens > 10)


def analyze_image_for_ada(image_path):
    """
    Run OCR + ADA checks on a single image.
    Returns a dict of metrics (print + digital relevant).
    """
    img_path = Path(image_path)
    img = Image.open(img_path).convert("RGB")
    img_np = np.array(img)

    h, w, _ = img_np.shape

    # --- Print DPI estimate ---
    est_dpi = w / ASSUMED_PRINT_WIDTH_IN
    dpi_ok = est_dpi >= MIN_PRINT_DPI

    # OCR with detailed data
    ocr_data = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT)

    has_text = False
    text_boxes = []
    min_contrast = None
    min_text_height = None

    n = len(ocr_data["text"])
    for i in range(n):
        text = ocr_data["text"][i].strip()
        try:
            conf = int(float(ocr_data["conf"][i]))
        except ValueError:
            conf = -1

        if not text:
            continue
        if conf < OCR_CONF_THRESHOLD:
            continue

        has_text = True
        x = int(ocr_data["left"][i])
        y = int(ocr_data["top"][i])
        w_box = int(ocr_data["width"][i])
        h_box = int(ocr_data["height"][i])

        text_boxes.append({
            "text": text,
            "conf": conf,
            "bbox": [x, y, w_box, h_box]
        })

        # smallest text height
        if min_text_height is None or h_box < min_text_height:
            min_text_height = h_box

        # sample foreground (center of bbox)
        cx = min(max(x + w_box // 2, 0), w - 1)
        cy = min(max(y + h_box // 2, 0), h - 1)
        fg_color = img_np[cy, cx, :]

        # sample background above text if possible
        by = max(y - 2, 0)
        bx = cx
        bg_color = img_np[by, bx, :]

        cr = contrast_ratio(fg_color, bg_color)
        if min_contrast is None or cr < min_contrast:
            min_contrast = cr

    # Blur detection via variance of Laplacian
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    blur_metric = lap.var()

    # Colorblind risk: compare original vs deuteranopia simulation
    cb_sim = simulate_colorblind_deuteranopia(img_np)
    # Compare average color difference
    diff = np.mean(np.abs(img_np.astype(float) - cb_sim.astype(float)))
    colorblind_risk = diff < 10.0  # low difference -> colors collapse somewhat

    # Grayscale risk: large collapse of distinct colors to same luminance
    gray_unique = len(np.unique(gray))
    color_unique = len(np.unique(img_np.reshape(-1, 3), axis=0))
    if color_unique == 0:
        grayscale_risk = False
    else:
        grayscale_risk = (gray_unique / color_unique) < 0.2

    # Table-like and complexity
    table_like = estimate_table_like(text_boxes) if has_text else False
    complex_figure = estimate_complex_figure(text_boxes) if has_text else False

    # Build result
    result = {
        "image_file": img_path.name,
        "has_text": bool(has_text),
        "min_contrast": float(min_contrast) if min_contrast is not None else None,
        "min_text_height_px": int(min_text_height) if min_text_height is not None else None,
        "blur_metric": float(blur_metric),
        "estimated_dpi": float(est_dpi),
        "dpi_ok": bool(dpi_ok),
        "colorblind_risk": bool(colorblind_risk),
        "grayscale_risk": bool(grayscale_risk),
        "table_like": bool(table_like),
        "complex_figure": bool(complex_figure),
        "ocr_text_boxes": text_boxes
    }

    # Print-focused flags
    contrast_ok = (min_contrast is None) or (min_contrast >= MIN_CONTRAST_PASS)
    text_size_ok = (min_text_height is None) or (min_text_height >= MIN_TEXT_HEIGHT_PX)
    blur_ok = blur_metric >= BLUR_THRESHOLD

    result["contrast_ok"] = bool(contrast_ok)
    result["text_size_ok"] = bool(text_size_ok)
    result["blur_ok"] = bool(blur_ok)
    result["print_pass"] = bool(contrast_ok and text_size_ok and blur_ok and dpi_ok)

    return result


# =========================
# MODULE 3: Alt text evaluation (digital ADA)
# =========================

def evaluate_alt_text(alt_text, has_text, table_like, complex_figure):
    """
    Evaluate alt text quality and digital ADA implications.
    Returns dict with score and flags.
    """
    alt = (alt_text or "").strip()
    has_alt = len(alt) > 0

    # basic quality checks
    words = alt.split()
    word_count = len(words)
    too_short = has_alt and word_count < ALT_MIN_WORDS
    too_long = has_alt and len(alt) > ALT_MAX_CHARS

    # generic / low-quality alt text
    lower = alt.lower()
    generic_patterns = [
        "image", "picture", "photo", "graphic", "figure", "chart"
    ]
    generic_only = False
    if has_alt:
        # e.g. "image", "chart 1", "figure 2"
        if word_count <= 3 and any(p in lower for p in generic_patterns):
            generic_only = True

    # should this image be informational?
    informational = has_text or table_like or complex_figure

    # decorative candidate: no text, no table, not complex
    decorative_candidate = not informational

    # does this image need a long description?
    long_desc_needed = informational and complex_figure

    # digital pass logic:
    # - if informational: must have alt text, not generic, not too short
    # - if decorative: alt can be empty (but that decision is manual)
    if informational:
        digital_pass = has_alt and not generic_only and not too_short and not too_long
    else:
        # decorative images okay even with empty alt (but we can't enforce tagging here)
        digital_pass = True

    return {
        "alt_present": has_alt,
        "alt_too_short": bool(too_short),
        "alt_too_long": bool(too_long),
        "alt_generic": bool(generic_only),
        "informational": bool(informational),
        "decorative_candidate": bool(decorative_candidate),
        "long_desc_needed": bool(long_desc_needed),
        "digital_pass": bool(digital_pass)
    }


# =========================
# MODULE 4: Reporting (CSV, JSON, HTML)
# =========================

def write_reports(out_dir, results):
    out_dir = Path(out_dir)

    # CSV
    csv_path = out_dir / "ada_report.csv"
    fieldnames = [
        "image_file",
        "alt_text",
        "has_text",
        "min_contrast",
        "min_text_height_px",
        "blur_metric",
        "estimated_dpi",
        "dpi_ok",
        "contrast_ok",
        "text_size_ok",
        "blur_ok",
        "colorblind_risk",
        "grayscale_risk",
        "table_like",
        "complex_figure",
        "alt_present",
        "alt_too_short",
        "alt_too_long",
        "alt_generic",
        "informational",
        "decorative_candidate",
        "long_desc_needed",
        "print_pass",
        "digital_pass",
        "overall_ada_pass"
    ]
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in results:
            row = {k: r.get(k) for k in fieldnames}
            w.writerow(row)

    # JSON (full detail including OCR boxes)
    json_path = out_dir / "ada_report.json"
    safe_results = make_json_safe(results)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(safe_results, f, indent=2)

    # HTML
    html_path = out_dir / "ada_report.html"
    with open(html_path, "w", encoding="utf-8") as f:
        f.write("<html><head><meta charset='utf-8'><title>ADA Image Report</title>")
        f.write("<style>")
        f.write("body { font-family: Arial, sans-serif; }")
        f.write("table { border-collapse: collapse; width: 100%; }")
        f.write("th, td { border: 1px solid #ccc; padding: 4px; font-size: 12px; }")
        f.write("th { background-color: #eee; }")
        f.write(".ok { color: green; font-weight: bold; }")
        f.write(".fail { color: red; font-weight: bold; }")
        f.write("</style></head><body>\n")
        f.write("<h1>ADA Image Report</h1>\n")
        f.write("<table>\n")
        f.write("<tr>"
                "<th>Image</th>"
                "<th>Print ADA</th>"
                "<th>Digital ADA</th>"
                "<th>Contrast</th>"
                "<th>Text Size</th>"
                "<th>Blur</th>"
                "<th>DPI</th>"
                "<th>Colorblind Risk</th>"
                "<th>Grayscale Risk</th>"
                "<th>Alt Text</th>"
                "</tr>\n")

        for r in results:
            img_rel = f"media/{r['image_file']}"
            print_pass = r.get("print_pass")
            digital_pass = r.get("digital_pass")
            overall = r.get("overall_ada_pass")
            contrast_ok = r.get("contrast_ok")
            text_size_ok = r.get("text_size_ok")
            blur_ok = r.get("blur_ok")
            dpi_ok = r.get("dpi_ok")
            cb_risk = r.get("colorblind_risk")
            gs_risk = r.get("grayscale_risk")
            alt_text = (r.get("alt_text") or "").replace("<", "&lt;").replace(">", "&gt;")

            f.write("<tr>")
            f.write(f"<td><img src='{img_rel}' style='max-width:260px; max-height:180px;'><br>{r['image_file']}</td>")

            # Print ADA
            f.write("<td>")
            cls = "ok" if print_pass else "fail"
            f.write(f"<span class='{cls}'>{print_pass}</span>")
            f.write("</td>")

            # Digital ADA
            f.write("<td>")
            cls = "ok" if digital_pass else "fail"
            f.write(f"<span class='{cls}'>{digital_pass}</span>")
            f.write("</td>")

            # Contrast
            f.write("<td>")
            if r.get("min_contrast") is None:
                f.write("n/a")
            else:
                cls = "ok" if contrast_ok else "fail"
                f.write(f"<span class='{cls}'>{r['min_contrast']:.2f}</span>")
            f.write("</td>")

            # Text size
            f.write("<td>")
            if r.get("min_text_height_px") is None:
                f.write("n/a")
            else:
                cls = "ok" if text_size_ok else "fail"
                f.write(f"<span class='{cls}'>{r['min_text_height_px']} px</span>")
            f.write("</td>")

            # Blur
            f.write("<td>")
            cls = "ok" if blur_ok else "fail"
            f.write(f"<span class='{cls}'>{r['blur_metric']:.1f}</span>")
            f.write("</td>")

            # DPI
            f.write("<td>")
            cls = "ok" if dpi_ok else "fail"
            f.write(f"<span class='{cls}'>{r['estimated_dpi']:.0f} dpi</span>")
            f.write("</td>")

            # Colorblind risk
            f.write("<td>")
            f.write("Risk" if cb_risk else "OK")
            f.write("</td>")

            # Grayscale risk
            f.write("<td>")
            f.write("Risk" if gs_risk else "OK")
            f.write("</td>")

            # Alt text
            f.write(f"<td>{alt_text}</td>")

            f.write("</tr>\n")

        f.write("</table>\n</body></html>\n")

    print("Wrote:", csv_path)
    print("Wrote:", json_path)
    print("Wrote:", html_path)


# =========================
# MODULE 5 & 6: Orchestrator (print + digital scoring)
# =========================

def run_pipeline_for_docx(docx_file, output_root):
    docx_file = Path(docx_file)
    output_root = Path(output_root)
    doc_out_dir = output_root / docx_file.stem
    doc_out_dir.mkdir(parents=True, exist_ok=True)

    # 1) Extract images
    extracted, media_dir = extract_images(docx_file, doc_out_dir)

    if not extracted:
        print("No images found in document.")
        return

    # 2) Alt text map
    alt_map = extract_alt_text_map(docx_file)

    # 3) ADA analysis per image
    results = []
    for img_name in extracted:
        img_path = media_dir / img_name
        ada = analyze_image_for_ada(img_path)
        alt_text = alt_map.get(img_name, "")
        ada["alt_text"] = alt_text

        # Alt text evaluation (digital)
        alt_eval = evaluate_alt_text(
            alt_text=alt_text,
            has_text=ada["has_text"],
            table_like=ada["table_like"],
            complex_figure=ada["complex_figure"]
        )
        ada.update(alt_eval)

        # Overall ADA pass: must satisfy both print & digital
        ada["overall_ada_pass"] = bool(ada["print_pass"] and ada["digital_pass"])

        results.append(ada)

    # 4) Write reports
    write_reports(doc_out_dir, results)

    print("\nPipeline complete for:", docx_file)
    print("Output folder:", doc_out_dir)


# =========================
# MAIN
# =========================

if __name__ == "__main__":
    # ✏️ EDIT THESE PATHS BEFORE RUNNING
    
    DOCX_FILE =  r"\\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25.docx"
    OUTPUT_ROOT = r"\\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports"

    run_pipeline_for_docx(DOCX_FILE, OUTPUT_ROOT)


Extracted: word/media/image1.png -> \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25\media\image1.png
Extracted: word/media/image2.png -> \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25\media\image2.png
Extracted: word/media/image3.png -> \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25\media\image3.png
Extracted: word/media/image4.png -> \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25\media\image4.png
Extracted: word/media/image5.png -> \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25\media\image5.png
Extracted: word/media/image6.png -> \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\BTS_Port-Performance-2026_Annual-Repor